In [ ]:

# Glomeruli Denoising - Main Notebook

# Cell 1: Imports
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.functional import pairwise_distance
from torch.optim.lr_scheduler import StepLR
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix
import seaborn as sns
from torch.utils.data import Dataset
import random
import os
import re
import json
import cv2
import wandb

# Import our custom functions
from src.data_processing import load_glomeruli_data
from src.dataset import GlomeruliContrastiveDataset
from src.utils import calculate_dataset_stats

print(torch.__version__)
print(torchvision.__version__)








In [ ]:
# Cell 2: Data Loading
# Define the paths to save the datasets
noisy_train_path = "path to train dataset"
noisy_test_path = "path to test dataset"

# Load data using our function
train_data, train_labels, test_data, test_labels = load_glomeruli_data(
    noisy_train_path, noisy_test_path
)

In [ ]:
# Cell 3: Dataset Statistics
# Calculate statistics for train data
train_stats = calculate_dataset_stats(train_data)
print("Train Dataset Statistics:")
for key, value in train_stats.items():
    print(f"{key.capitalize()}: {value}")

# Calculate statistics for test data
test_stats = calculate_dataset_stats(test_data)
print("\nTest Dataset Statistics:")
for key, value in test_stats.items():
    print(f"{key.capitalize()}: {value}")



In [ ]:
#  noisy_train_data and noisy_train_labels are glomeruli activation data and labels

noisy_glomeruli_dataset = GlomeruliContrastiveDataset(noisy_train_data, noisy_train_labels, is_training=True)
train_loader = DataLoader(noisy_glomeruli_dataset, batch_size=32, shuffle=True)

#  noisy_train_data and noisy_train_labels are  glomeruli activation data and labels

noisy_glomeruli_dataset_test = GlomeruliContrastiveDataset(noisy_test_data, noisy_test_labels, is_training=False)
test_loader = DataLoader(noisy_glomeruli_dataset_test, batch_size=32, shuffle=True)


In [ ]:
# Cell 5: Dataset Sanity Check
from src.utils import dataset_sanity_check

# Test the dataset
dataset_sanity_check(train_dataset, sample_index=15)

In [ ]:
# Cell 6: Visualize Contrastive Samples
from src.utils import visualize_contrastive_samples

# Visualize samples
visualize_contrastive_samples(train_dataset, sample_index=15)

In [ ]:
# Cell 7: Model Definition
from src.model import get_model

# Choose your model - now you have 3 options!
model_choice = "unet"          # U-Net with skip connections (best for denoising)
# model_choice = "basic_cnn"     # Simple CNN (fastest)
# model_choice = "simple_cnn"    # CNN with batch norm (middle option)

model = get_model(model_choice)

print(f"Selected model: {model_choice}")
print(f"Model class: {model.__class__.__name__}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test model with correct input/output shapes
test_input = torch.randn(1, 1, 256, 256)  # Batch=1, Channels=1, Height=256, Width=256
test_output = model(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {test_output.shape}")



In [ ]:
print(torch.cuda.is_available())  # Should return True if GPU is available
print(torch.cuda.device_count())  # Should return the number of available GPUs
print(torch.cuda.current_device())  # Prints the index of the current GPU being used
print(torch.cuda.get_device_name(0))  # Prints the name of GPU 0


In [ ]:
# Cell 9: Training Setup
import yaml

# Load hyperparameters from config file
with open('configs/training_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize networks - using config
model_choice = config['training']['model_choice']
net_F = get_model(model_choice).to(device)
net_B = get_model(model_choice).to(device)

# Initialize optimizers - using config
lr = config['training']['learning_rate']
optimizer = optim.Adam(list(net_F.parameters()) + list(net_B.parameters()), lr=lr)

# Optional: Other optimizers from config
if config['training'].get('use_adamw', False):
    optimizer = optim.AdamW(list(net_F.parameters()) + list(net_B.parameters()), 
                           lr=lr, 
                           weight_decay=config['training']['weight_decay'])

In [ ]:
# Cell 10: Loss Weights and Experiment Tracking
# Initialize a W&B run (uncomment when ready)
#wandb.init(project=config['training']['wandb_project'], 
#           name=config['training']['wandb_run_name'])

# Load loss weights from config
weight_recon = config['training']['weight_recon']
weight_subject = config['training']['weight_subject'] 
weight_stimulus = config['training']['weight_stimulus']
weight_contrast = config['training']['weight_contrast']

print(f"Loss weights - Recon: {weight_recon}, Subject: {weight_subject}, Stimulus: {weight_stimulus}, Contrast: {weight_contrast}")

In [ ]:
# Import the function
from src.utils import save_checkpoint

#  during training, call it like this:
save_checkpoint(epoch, net_F, net_B, optimizer, loss)

# Or with custom save directory:
save_checkpoint(epoch, net_F, net_B, optimizer, loss, 
                save_dir="/your/custom/path")

In [ ]:
# Cell 11: Main Training Loop
import os
from src.training import train_epoch, validate_epoch
from src.utils import save_checkpoint, plot_denoising_results, plot_training_curves

# Create checkpoint directory
os.makedirs(config['training']['checkpoint_dir'], exist_ok=True)

# Initialize tracking
num_epochs = config['training']['num_epochs']
train_losses = []
test_losses = []

# Prepare loss weights dictionary
weights = {
    'recon': config['training']['weight_recon'],
    'subject': config['training']['weight_subject'],
    'stimulus': config['training']['weight_stimulus'],
    'contrast': config['training']['weight_contrast']
}

print(f"Starting training for {num_epochs} epochs...")
print(f"Loss weights: {weights}")

# Main training loop
for epoch in range(num_epochs):
    print(f"\n=== Epoch {epoch+1}/{num_epochs} ===")
    
    # Training phase
    train_loss = train_epoch(
        net_F, net_B, train_loader, optimizer, device, weights,
        print_freq=config['training']['print_frequency']
    )
    
    # Validation phase
    test_loss = validate_epoch(net_F, net_B, test_loader, device, weights)
    
    # Store losses
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    
    print(f"Epoch {epoch+1} Complete: Train Loss: {train_loss:.6f}, Test Loss: {test_loss:.6f}")
    
    # Save checkpoint
    if epoch % config['training']['checkpoint_frequency'] == 0:
        save_checkpoint(epoch, net_F, net_B, optimizer, train_loss, 
                       save_dir=config['training']['checkpoint_dir'])
    
    # Plot results
    if (epoch + 1) % config['training']['plot_frequency'] == 0:
        plot_denoising_results(net_F, net_B, test_loader, device,
                              num_images=config['training']['num_images_to_plot'])

print("Training complete!")

# Plot final training curves
plot_training_curves(train_losses, test_losses, num_epochs)

In [ ]:
# Cell 12: Specific Image Analysis
from src.utils import plot_specific_odorant_image

# Example usage
odorant_idx = 1  
image_idx = 1    
plot_specific_odorant_image(odorant_idx, image_idx, net_F, net_B, test_loader, device)

In [ ]:
# Cell 13: Checkpoint Management
from src.utils import get_sorted_checkpoints

# Get sorted checkpoints from config directory
checkpoints = get_sorted_checkpoints(config['training']['checkpoint_dir'])
print("Available checkpoints:", len(checkpoints))
if checkpoints:
    print("Latest checkpoint:", checkpoints[-1])

In [ ]:
# Cell 14: Load Validation Data
from src.data_processing import load_coco_annotations

# Load COCO annotations
data_path = "/gpfs/scratch/agarwv03/merged_dataset/raw_validation_images/"
image_info, annotations_info = load_coco_annotations(data_path)

In [ ]:
# After loading COCO data
from src.data_processing import create_image_annotation_mapping

# Create the mapping
image_id_to_annotations = create_image_annotation_mapping(annotations_info)

# Verify the mapping
sample_image_id = 1
print(f"Annotations for Image ID {sample_image_id}:", image_id_to_annotations[sample_image_id])

In [ ]:
# Load images with annotations
images = load_images_with_annotations(image_info, image_id_to_annotations, data_path)

# Verify
sample_image_id = 1
print("Sample Image Data:", images[sample_image_id])

In [ ]:
# Calculate R metrics for all images
from src.data_processing import calculate_r_metrics

r_metrics = calculate_r_metrics(images)

In [ ]:
# Visualize all images with their masks
from src.utils import visualize_image_and_mask

for image_id, data in images.items():
    image = data['image']
    bboxes = [ann['bbox'] for ann in data['bboxes']]
    visualize_image_and_mask(image, bboxes, image_id)

In [ ]:
# Cell 15: Load Pretrained Model
from src.model import get_model
from src.utils import load_checkpoint

# Reinitialize models using your model factory
net_F = get_model(model_choice).to(device)
net_B = get_model(model_choice).to(device)

# Load specific checkpoint
checkpoint_path = "/gpfs/data/rinberglab/vivek/checkpoints___alt___/checkpoint_epoch_38.pth"
checkpoint = load_checkpoint(checkpoint_path, net_F, net_B, device)

# Access checkpoint data
loaded_epoch = checkpoint['epoch']
loaded_loss = checkpoint['loss']
print(f"Loaded Epoch: {loaded_epoch}, Loss: {loaded_loss}")

In [ ]:
# Cell 16: Single Image Inference Test
# Retrieve the already normalized image
image_id = 9  
sample_image = images[image_id]['image']

# Add batch and channel dimensions for model input
input_tensor = torch.tensor(sample_image, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

# Pass the image through networks
with torch.no_grad():
    denoised_output = net_F(input_tensor)
    background_image = net_B(input_tensor)

# Check the output shape and range
print(f"Denoised Output Shape: {denoised_output.shape}")
print(f"Denoised Output Range: Min={denoised_output.min().item()}, Max={denoised_output.max().item()}")

In [ ]:
# Convert the denoised output to a numpy array for visualization
denoised_image = denoised_output.squeeze().cpu().numpy()
Background_image = background_image.squeeze().cpu().numpy()

# Plot the images
plt.figure(figsize=(12, 6))

# Original Image
plt.subplot(1, 3, 1)
plt.imshow(sample_image, cmap='gray')
plt.title(f"Image ID {image_id}: Original")
plt.axis('off')

# Denoised Image
plt.subplot(1, 3, 2)
plt.imshow(denoised_image, cmap='gray')
plt.title(f"Image ID {image_id}: Denoised")
plt.axis('off')

# Background Image
plt.subplot(1, 3, 3)
plt.imshow(Background_image, cmap='gray')
plt.title(f"Image ID {image_id}: Background_image")
plt.axis('off')

plt.show()


In [ ]:
# Apply to the denoised image
from src.data_processing import calculate_r_metric_single

bboxes = [ann['bbox'] for ann in images[image_id]['bboxes']]
denoised_image_np = denoised_output.squeeze().cpu().numpy()  # Convert tensor to numpy

r_metric = calculate_r_metric_single(denoised_image_np, bboxes)
print(f"R Metric for Image ID {image_id}: {r_metric:.4f}")

In [ ]:
# Cell 17: R Metric Analysis Across Checkpoints
from src.utils import get_sorted_checkpoints, calculate_r_for_checkpoint
from src.model import get_model

# Get checkpoints and filter every 10th epoch
checkpoints = get_sorted_checkpoints(config['training']['checkpoint_dir'])
# Filter for every 10th epoch or adjust as needed

# Initialize models with your model factory
net_F = get_model(model_choice).to(device)
net_B = get_model(model_choice).to(device)

# Calculate R metrics across checkpoints
r_trends = {odor: [] for odor in range(10)}
for checkpoint_path in checkpoints:
    r_metrics = calculate_r_for_checkpoint(checkpoint_path, net_F, net_B, images, device)
    for odor, r_metric in r_metrics.items():
        r_trends[odor].append(r_metric)

In [ ]:
# Plot the trends
from src.utils import plot_r_metric_trends

plot_r_metric_trends(r_trends, original_r_metrics, epoch_numbers)

In [ ]:
# Cell 18: ROI Ratio Analysis Across Checkpoints
import re
from src.utils import calculate_roi_ratios_for_checkpoint

# Dictionary to store ROI ratios for each odor across checkpoints
roi_ratio_trends = {odor: [] for odor in range(10)}
epoch_numbers = []

for ckpt_path in checkpoints:
    # Extract epoch number
    epoch_number = int(re.search(r'checkpoint_epoch_(\d+).pth', ckpt_path).group(1))
    epoch_numbers.append(epoch_number)
    
    # Calculate ROI ratios
    odor_roi_ratios = calculate_roi_ratios_for_checkpoint(
        ckpt_path, net_F, net_B, images, device
    )
    
    # Store ratios for trend analysis
    for odor_label, ratios_list in odor_roi_ratios.items():
        roi_ratio_trends[odor_label].append(ratios_list)

print(f"Processed {len(checkpoints)} checkpoints")
print(f"Epoch range: {min(epoch_numbers)} to {max(epoch_numbers)}")

In [ ]:
# Cell 19: Plot Mean ROI Ratios
from src.utils import plot_mean_roi_ratios

# Plot for a specific odor
plot_mean_roi_ratios(roi_ratio_trends, epoch_numbers, selected_odor=0)